# Phase 3c -- diagnostics micro-session (A100-40GB, High-RAM **OFF**)

Four sanity/bisection questions left open by Phase 3b, bundled into one
from-scratch session so setup is paid once (9 cells, 4 server launches,
~1.5-2h, ~20-25 units). Generator + interpretation grid:
`configs/diagnostics/generate_diagnostics.py`.

1. **tau-eager-short** -- is the probe's tau=1.14 a long-context finding or an
   eager artifact? (tau~2.85 here = finding is real)
2. **slong-eager-base** -- no-spec eager baseline at 7.4k context: measures
   S-main at long context within the eager regime (probe refs: 34.0 tok/s @ c1,
   166.4 @ c8).
3. **cudagraph-probe** -- compile ON, CUDA graphs OFF on the crashing corner.
   **A crash here is an informative outcome, not a failure of the session** --
   do not retry it; it feeds the upstream vLLM issue only.
4. **backendpin-flashinfer** -- FP16-KV c8 with FLASHINFER pinned: bounds the
   backend component inside every K contrast (k_stress ref: ~221 tok/s on
   FLASH_ATTN).

Push the repo before starting -- Colab pulls from GitHub.

In [1]:
# 1. Verify the 40GB card (High-RAM OFF)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import subprocess
gpu = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert "A100" in gpu, "Not an A100 - reconnect until Colab honors the selection."
mem = int(subprocess.run(["nvidia-smi","--query-gpu=memory.total","--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip().splitlines()[0])
assert mem < 50000, ("Got the 80GB A100 (%d MiB). Toggle High-RAM OFF and reconnect: "
                     "these diagnostics are compared against Phase-3b 40GB cells.") % mem
units_before = input("Compute-units balance right now: ")

NVIDIA A100-SXM4-40GB, 40960 MiB
Compute-units balance right now: 94.78


In [2]:
# 2. Repo + Spec-Bench (long documents are built from its RAG passages)
%cd /content
!git clone https://github.com/manojarulmurugan/SpecDecoding-Study-vLLM-SGLang.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo
!git clone --depth 1 https://github.com/hemingkx/Spec-Bench.git external/Spec-Bench 2>/dev/null || echo "Spec-Bench already present"
!test -f external/Spec-Bench/data/spec_bench/question.jsonl && echo "RAG data OK"

/content
/content/repo
RAG data OK


In [3]:
# 3. Isolated vLLM environment (PREREQ_RESULTS Check 6 recipe; ~6-8 min)
%pip install -q virtualenv
!virtualenv -q /content/vllm_env
!/content/vllm_env/bin/pip install -q vllm==0.24.0 datasets pyyaml requests pytest ninja
!/content/vllm_env/bin/python -c "import vllm; print('vllm', vllm.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 123.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 40.5 MB/s eta 0:00:00
vllm 0.24.0


In [5]:
# 4. HF auth
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token set:", bool(os.environ["HF_TOKEN"]))

token set: True


In [ ]:
# 4b. Pre-download checkpoints (bounded, retried; automatic browser-UA curl
# fallback if HF's hf-client CDN route is broken -- PREREQ 2026-07-14/15).
# If hf is KNOWN-broken right now, use:  ... predownload.py --curl-only
!/content/vllm_env/bin/python scripts/predownload.py

In [6]:
#4b Alt: restore the HF cache from the Phase 3b Drive backup instead
# (specdecoding_hf_cache -- the actual backup dir from the Phase 3b session,
# not a guessed path). Covers both checkpoints this phase needs.
from google.colab import drive
drive.mount('/content/drive')

backup_dir = "/content/drive/MyDrive/specdecoding_hf_cache"
!mkdir -p /root/.cache/huggingface
!rsync -a --info=progress2 {backup_dir}/ /root/.cache/huggingface/
print("Restored. Verify predownload sees everything cached:")
!/content/vllm_env/bin/python scripts/predownload.py


Mounted at /content/drive
 22,674,430,732  99%   68.16MB/s    0:05:17 (xfr#82, to-chk=0/145)
Restored. Verify predownload sees everything cached:
[predownload] meta-llama/Llama-3.1-8B-Instruct (timeout 1800s): /content/vllm_env/bin/hf download meta-llama/Llama-3.1-8B-Instruct

✓ Downloaded
  path: /root/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659


[predownload] hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4 (timeout 1800s): /content/vllm_env/bin/hf download hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4

✓ Downloaded
  path: /root/.cache/huggingface/hub/models--hugging-quants--Meta-Llama-3.1-8B-Instruct-AWQ-INT4/snapshots/db1f81ad4b8c7e39777509fac66c652eb0a52f91


[predownload] yuhuili/EAGLE3-LLaMA3.1-Instruct-8B (timeout 1800s): /content/vllm_env/bin/hf download yuhuili/EAGLE3-LLaMA3.1-Instruct-8B

✓ Downloaded
  path: /root/.cache/huggingface/hub/models--yuhuili--EAGLE3-LLaMA3.1-Instruct-8B/snapshots/ada

In [7]:
# 5. Harness self-check (~1 min, no GPU)
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m pytest tests -q

........................................................................ [ 40%]
........................................................................ [ 81%]
.................................                                        [100%]
177 passed in 16.83s


In [8]:
# 6. Sanity: 9 cells in 4 server groups; check the four distinct commands
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/diagnostics/diag_*.yaml" --results-dir results --dry-run 2>&1 | grep -E "server command|run\(s\)"

[sweep] 9 run(s) in 4 server group(s)
[sweep] group 1/4: 2 pending run(s)
[sweep] server command: vllm serve meta-llama/Llama-3.1-8B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.85 --max-model-len 8192 --seed 1234 --no-enable-prefix-caching
[sweep] group 2/4: 1 pending run(s)
[sweep] server command: vllm serve meta-llama/Llama-3.1-8B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.85 --max-model-len 8192 --seed 1234 --no-enable-prefix-caching --speculative-config {"draft_tensor_parallel_size": 1, "method": "eagle3", "model": "yuhuili/EAGLE3-LLaMA3.1-Instruct-8B", "num_speculative_tokens": 5} --max-num-batched-tokens 8192 --compilation-config {"cudagraph_mode": "NONE"}
[sweep] group 3/4: 4 pending run(s)
[sweep] server command: vllm serve meta-llama/Llama-3.1-8B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.85 --max-model-len 8192 --seed 1234 --no-enable-prefix-caching --max-num-batch

The τ-retest -- expanded to all 4 probe corners (c1 r0/r1, c8 r0/r1), not just the single cell from the first pass, for the same statistical confidence the original finding had.

In [9]:
import shutil, json, pathlib, glob, os

snap_dirs = glob.glob("/root/.cache/huggingface/hub/models--yuhuili--EAGLE3-LLaMA3.1-Instruct-8B/snapshots/*")
assert snap_dirs, "EAGLE3 checkpoint not found -- restore/download it first"
src = pathlib.Path(snap_dirs[0])

dst = pathlib.Path("/content/eagle3_draft_8192ctx")
dst.mkdir(parents=True, exist_ok=True)
for f in src.iterdir():
    target = dst / f.name
    if target.exists():
        continue
    if f.name == "config.json":
        shutil.copy(f.resolve(), target)   # real editable copy
    else:
        os.symlink(f.resolve(), target)    # weights: just link, no need to duplicate GBs

cfg_path = dst / "config.json"
cfg = json.loads(cfg_path.read_text())
print("before:", cfg.get("max_position_embeddings"))
cfg["max_position_embeddings"] = 8192
cfg_path.write_text(json.dumps(cfg, indent=2))
print("after:", json.loads(cfg_path.read_text())["max_position_embeddings"])

before: 2048
after: 8192


In [10]:
import pathlib

pathlib.Path("/content/retest_configs").mkdir(exist_ok=True)
for name in ["kstress_eagle3-fp16kv_c1_r0.yaml", "kstress_eagle3-fp16kv_c1_r1.yaml",
             "kstress_eagle3-fp16kv_c8_r0.yaml", "kstress_eagle3-fp16kv_c8_r1.yaml"]:
    text = pathlib.Path(f"configs/k_stress/{name}").read_text()
    text = text.replace(
        "draft_model: yuhuili/EAGLE3-LLaMA3.1-Instruct-8B",
        "draft_model: /content/eagle3_draft_8192ctx",
    )
    text = text.replace(
        'extra: ["--max-num-batched-tokens", "8192", "--enforce-eager"]',
        'extra: ["--max-num-batched-tokens", "8192"]',
    )
    dst = pathlib.Path("/content/retest_configs") / name
    dst.write_text(text)
    print(dst, "written")


/content/retest_configs/kstress_eagle3-fp16kv_c1_r0.yaml written
/content/retest_configs/kstress_eagle3-fp16kv_c1_r1.yaml written
/content/retest_configs/kstress_eagle3-fp16kv_c8_r0.yaml written
/content/retest_configs/kstress_eagle3-fp16kv_c8_r1.yaml written


In [11]:
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "/content/retest_configs/*.yaml" --results-dir /content/retest_results_full


[sweep] 4 run(s) in 1 server group(s)
[sweep] group 1/1: 4 pending run(s)
[sweep] server command: vllm serve meta-llama/Llama-3.1-8B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.85 --max-model-len 8192 --seed 1234 --no-enable-prefix-caching --speculative-config {"draft_tensor_parallel_size": 1, "method": "eagle3", "model": "/content/eagle3_draft_8192ctx", "num_speculative_tokens": 5} --max-num-batched-tokens 8192
[sweep] waiting for server (log: /content/retest_results_full/server_logs/server_20260717_235455.log)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[run] k_stress_llama-3-1-8

In [12]:
# RoPE OOB instrumentation (analysis/vllm_2048_bug_diagnosis.md appendix):
# empirically settles what eager mode's rope does at positions >= 2048 --
# resolved custom_ops + actual dispatch target, forward_cuda values vs an
# independent math reference (8192-cache control included), and whether
# forward_native raises. Each probe runs in its own subprocess (device
# asserts poison a CUDA context). ~2 min. Paste the FULL output back into
# the session; the GitHub draft comment stays on hold until then.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python scripts/debug_rope_oob.py

==================== probe: dispatch (exit 1)
Traceback (most recent call last):
  File "/content/repo/scripts/debug_rope_oob.py", line 206, in <module>
    sys.exit(main())
             ^^^^^^
  File "/content/repo/scripts/debug_rope_oob.py", line 160, in main
    print(json.dumps(PROBES[args.probe](), indent=1))
                     ^^^^^^^^^^^^^^^^^^^^
  File "/content/repo/scripts/debug_rope_oob.py", line 102, in probe_dispatch
    "enabled": type(rope).enabled(),
               ^^^^^^^^^^^^^^^^^^^^
  File "/content/vllm_env/lib/python3.12/site-packages/vllm/model_executor/custom_op.py", line 274, in enabled
    compilation_config = get_cached_compilation_config()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/vllm_env/lib/python3.12/site-packages/vllm/config/vllm.py", line 2249, in get_cached_compilation_config
    return get_current_vllm_config().compilation_config
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/vllm_env/lib/python3.12/site-packag

In [13]:
!zip -qr retest_results_full.zip /content/retest_results_full
from google.colab import files
files.download("retest_results_full.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Compute Units left --> Available: 92.37 compute units

-----